# Extending an Existing Model

This notebook shows how to subclass one of bacpipe's built-in models to modify
its behaviour — for example to change the embedding extraction or the
preprocessing — while keeping the surrounding pipeline (audio loading, windowing,
saving, probing) untouched.


Make sure to reset your config and settings files before running this notebook, 
as they may contain settings from previous runs that could prevent this notebook 
from finding the correct paths.


---
## 1. Setup & Configuration
Import necessary modules 

In [1]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython

from IPython.display import display
import os 
from pathlib import Path

# load the specific package
import bacpipe

/home/siriussound/Code/testing_repos/bacpipe/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/siriussound/Code/testing_repos/bacpipe/bacpipe/embedding_evaluation/visualization/dashboard.py:42: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("plotly")


Set the working directory to the repository root and clean the previous tests if needed/wanted

In [2]:
import importlib.resources as pkg_resources
os.chdir(pkg_resources.files("bacpipe"))
os.chdir('..')


# Change the value of the key main_results_dir in the namespace bacpipe.settings to change the directory 
# where the results of the tutorials are stored. By default, it is set to './bacpipe_results'.
bacpipe.settings.main_results_dir = str(Path(bacpipe.settings.main_results_dir) / 'extending_an_existing_model')

# !WARNING! the following code deletes the folder where the results of this tutorial is stored to be sure to start with a clean folder. 
# If you have important data in this folder, please comment it before running this code.
folder_path = bacpipe.settings.main_results_dir
if os.path.exists(folder_path):
    # Prompt the user
    user_input = input(f"Are you sure you want to delete '{folder_path}'? (y/n): ").lower().strip()

    if user_input == 'y':
        import shutil
        shutil.rmtree(folder_path) 
        print(f"Folder {folder_path} deleted.")
    else:
        print("Operation cancelled.")

else:
    print(f"Folder {folder_path} not found.")

Folder bacpipe_results/extending_an_existing_model not found.


Set the global constants used throughout the notebook.

In [3]:
MODEL_NAME = 'custom_birdnet'           # Choose the name of the custom model to run.
AUDIO_DIR = 'bacpipe/tests/test_data'   # path to directory containing audio files

`MODEL_NAME` is the key under which the results of the modified model will be
stored, and `AUDIO_DIR` points to the audio used in this tutorial.


---
## 2. Extending an Existing Model

Subclass an existing bacpipe model to modify its behaviour — for example,
squaring the input before passing it through BirdNET. Each built-in model lives
in `bacpipe.model_pipelines.feature_extractors.<model_name>` and exposes a
`Model` class. The cell below lists all supported models and imports BirdNET's
`Model` class.

The subclass inherits everything from the parent — the preprocessing, the sample
rate, the segment length, the checkpoint handling — and only overrides what you
need. Here `__call__` receives the preprocessed audio tensor and forwards it to
`self.embeds(input, training=False)`, which runs BirdNET and returns the
embeddings. You could equally well change the preprocessing, average embeddings
over time, add a projection layer, etc.

To see what the base class provides, have a look at
`bacpipe/model_pipelines/feature_extractors/birdnet.py` and
`bacpipe/model_pipelines/model_utils.py`.


In [4]:
# Select the class Model of the desired model you would like to modify, here birdnet, but it could be another model in the list.
# To know all supported models, run bacpipe.supported_models.
# For instance, if one wants to use birdMAE, the code is :
# from bacpipe.model_pipelines.feature_extractors.birdmae import Model

display(bacpipe.supported_models)

from bacpipe.model_pipelines.feature_extractors.birdnet import Model

# Create a subclass from the class Model depending on the chosen model
class MyBirdNETModel(Model):
    # input here is the preprocessed audio. The preprocessing is model dependent. Have a look at the birdnet.py file to inspect in more detail
    def __call__(self, input):
        input = input
        return self.embeds(input, training=False)

['audiomae',
 'audioprotopnet',
 'avesecho_passt',
 'aves_especies',
 'bat',
 'batdetect2_clip_avg',
 'batdetect2_dets_avg',
 'beats',
 'birdaves_especies',
 'biolingual',
 'birdnet_v3',
 'birdnet',
 'birdmae',
 'convnext_birdset',
 'hbdet',
 'insect66',
 'insect459',
 'mix2',
 'naturebeats',
 'perch_bird',
 'perch_v2',
 'protoclr',
 'rcl_fs_bsed',
 'surfperch',
 'google_whale',
 'vggish']

2026-08-26 16:49:23.477967: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-26 16:49:23.699227: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


---
## 3. Using this new built-in model

1. First generate the embeddings with the modified BirdNET model.
2. Train a probe to evaluate the quality of the embeddings of the model.

`run_pipeline_for_single_model` receives the subclass through the `CustomModel`
keyword argument; everything else works exactly like with the built-in models.
`ground_truth_by_model` connects the embeddings to the annotations,
`probing_pipeline` trains and evaluates a linear probe on the embeddings, and
`metrics['overall']` summarizes the probe performance.


In [5]:
# load the data to be process as well the model and compute the embeddings
loader_obj = bacpipe.run_pipeline_for_single_model(
    model_name=MODEL_NAME,                               # name of the model to run. Supported models are in bacpipe.supported_models
    audio_dir=AUDIO_DIR,                # path to directory containing audio files  
    CustomModel=MyBirdNETModel
)

# get the computed embeddings as an array
embeds = loader_obj.embeddings(return_type='array')

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.




###### Generating embeddings using CUSTOM_BIRDNET ######

finding audio files: 11it [00:00, 6465.43it/s]
Found 7 number of audio files.
Using device='cpu'
2026-08-26 16:49:25.431722: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-08-26 16:49:25.431998: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
2026-08-26 16:49:25.432003: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2026-08-26 16:49:25.432008: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic o

In [6]:
# Run this function after computing the embeddings otherwise it is no able to find the connection between embeddings and labels
gt = bacpipe.ground_truth_by_model(
    model=MODEL_NAME, 
    audio_dir=AUDIO_DIR, 
    annotations_filename='annotations.csv',
    overwrite=False
)

# Train and test a new probe associated to the feature vectors of the model 
# and evaluate the performance of the probe. 
# The returned metrics are the same as for the evaluation of a classification model, 
# but here they are used to evaluate the quality of the embeddings of the model.
probe, label2idx, metrics = bacpipe.probing_pipeline(
    model_name=MODEL_NAME, 
    ground_truth=gt,
    embeds=embeds)

# display the main metrics of the probe
display(metrics['overall'])

                                                                                                  
The simultaneous labels column of the ground truth has values exceeding 1. This means you have multi-label ground truth annotations. If this should not be happening ensure the ground truth is created correcly.

                                                                                                  
The simultaneous labels column of the ground truth has values exceeding 1. This means you have multi-label ground truth annotations. If this should not be happening ensure the ground truth is created correcly.

getting time of day: 100%|██████████| 7/7 [00:00<00:00, 151340.87it/s]
getting time per embeddings: 7it [00:00, 11925.32it/s]
getting day of year: 100%|██████████| 7/7 [00:00<00:00, 120823.57it/s]
getting continuous timestamps: 7it [00:00, 23620.38it/s]
getting parent directory: 7it [00:00, 23582.43it/s]
getting audio file names: 7it [00:00, 21992.61it/s]
Building metadata labe

{'macro_accuracy': 0.75, 'auc': 0.81875, 'macro_f1': 0.7142857142857143}

---
## 4. Full Pipeline - Multiple Models

`run_pipeline_for_models` runs the full pipeline across several models in one
call. Custom models are passed as a list through the `CustomModels` keyword, one
entry per model name and in the same order as `models` — `None` marks a built-in
model.

Because the subclassed BirdNET still produces a 1D embedding vector per window,
dimensionality reduction works here as well, so `umap` is used for the
visualization.

In [7]:
# combine the modified BirdNET with a built-in model: CustomModels is parallel to models
loader_dictionary = bacpipe.run_pipeline_for_models(
    models=[MODEL_NAME, 'perch_bird'],                  # modified BirdNET + built-in model
    audio_dir=AUDIO_DIR,                                # path to directory containing audio files
    dim_reduction_model='umap',                         # dimensionality reduction model to use for visualization
    CustomModels=[MyBirdNETModel, None],                # one class per model; None for built-in models
)

# the returned dictionary is keyed by model name and holds a Loader per model
display(loader_dictionary[MODEL_NAME].metadata_dict)

# each Loader exposes the same methods as before
print('custom_birdnet embeddings shape:', loader_dictionary[MODEL_NAME].embeddings(return_type='array').shape)
print('perch_bird embeddings shape:    ', loader_dictionary['perch_bird'].embeddings(return_type='array').shape)

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.




###### Generating embeddings using CUSTOM_BIRDNET ######

finding audio files: 11it [00:00, 5535.37it/s]
Found 7 number of audio files.
Using device='cpu'
Skipping model.eval() because model is from tensorflow.
 processing batches: 100%|██████████| 5/5 [00:01<00:00,  5.07it/s]2026-08-26 16:49:37.898967: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
                                                                  Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.




###### Generating embeddings using UMAP ######

Using device='cpu'
/home/siriussound/Code/testing_repos/bacpipe/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
umap inference took 6.42s

{'model_name': 'custom_birdnet',
 'audio_dir': 'bacpipe/tests/test_data',
 'embed_dir': 'bacpipe_results/extending_an_existing_model/test_data/embeddings/2026-08-26_16-49___custom_birdnet-test_data',
 'files': {'audio_files': ['audio/FewShot/CHE_01_20190101_163410.wav',
   'audio/FewShot/CHE_02_20190101_183410.wav',
   'audio/FewShot/CHE_03_20190201_163410.wav',
   'audio/FewShot/CHE_04_20190203_175410.wav',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031300.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031400.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031500.WAV'],
  'file_lengths (s)': [63.98977083333333,
   9.890729166666667,
   8.202479166666667,
   9.351708333333333,
   30.0,
   30.0,
   30.0],
  'nr_embeds_per_file': [22, 4, 3, 4, 10, 10, 10]},
 'segment_length (samples)': 144000,
 'sample_rate (Hz)': 48000,
 'embedding_size': 1024,
 'nr_embeds_total': 63,
 'total_dataset_length (s)': 181.4346875}

custom_birdnet embeddings shape: (63, 1024)
perch_bird embeddings shape:     (37, 1280)


---
## 5. End-to-End: `bacpipe.play`

`bacpipe.play` is the highest-level entry point and runs the full pipeline —
embeddings, classification, optional dimensionality reduction, evaluation and an
interactive dashboard — in a single call. Custom models are passed through the
`CustomModels` keyword, exactly as in `run_pipeline_for_models`.

The dashboard is disabled here (`dashboard=False`) so that the notebook runs
headlessly; set it to `True` to launch the interactive dashboard at
http://localhost:5006. Since the embeddings and the `umap` projections were
already computed in the previous section, this rerun only loads them and runs
the cross-model evaluation.

In [8]:
bacpipe.play(
    models=[MODEL_NAME, 'perch_bird'],   # modified BirdNET + built-in model
    audio_dir=AUDIO_DIR,                 # path to directory containing audio files
    dim_reduction_model='umap',          # dimensionality reduction model to use for visualization
    CustomModels=[MyBirdNETModel, None], # one class per model; None for built-in models
    dashboard=False,                     # set to True to launch the interactive dashboard
)

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

INFO:bacpipe:Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.



---
## 6. High-Level Workflow - `generate_embeddings`

`bacpipe.generate_embeddings` is the single-model building block behind
`run_pipeline_for_single_model` and `run_pipeline_for_models`. It accepts the
custom class through the `CustomModel` keyword as well. Because the embeddings
were already computed above, this call simply loads them from disk.

In [9]:
loader_obj = bacpipe.generate_embeddings(
    model_name=MODEL_NAME,
    audio_dir=AUDIO_DIR,
    CustomModel=MyBirdNETModel,
)

display(loader_obj.metadata_dict)

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

INFO:bacpipe:Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.




###### Generating embeddings using CUSTOM_BIRDNET ######

INFO:bacpipe:


###### Generating embeddings using CUSTOM_BIRDNET ######

Finding all generated embeddings: 7it [00:00, 22141.88it/s]
Found 7 embedding files.
INFO:bacpipe:Found 7 embedding files.
finding audio files: 11it [00:00, 29670.32it/s]
Found 7 number of audio files.
INFO:bacpipe:Found 7 number of audio files.

### Embeddings already exist. Using embeddings in bacpipe_results/extending_an_existing_model/test_data/embeddings/2026-08-26_16-49___custom_birdnet-test_data ###
INFO:bacpipe:
### Embeddings already exist. Using embeddings in bacpipe_results/extending_an_existing_model/test_data/embeddings/2026-08-26_16-49___custom_birdnet-test_data ###


{'audio_dir': 'bacpipe/tests/test_data',
 'embed_dir': 'bacpipe_results/extending_an_existing_model/test_data/embeddings/2026-08-26_16-49___custom_birdnet-test_data',
 'embedding_size': 1024,
 'files': {'audio_files': ['audio/FewShot/CHE_01_20190101_163410.wav',
   'audio/FewShot/CHE_02_20190101_183410.wav',
   'audio/FewShot/CHE_03_20190201_163410.wav',
   'audio/FewShot/CHE_04_20190203_175410.wav',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031300.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031400.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031500.WAV'],
  'file_lengths (s)': [63.98977083333333,
   9.890729166666667,
   8.202479166666667,
   9.351708333333333,
   30.0,
   30.0,
   30.0],
  'nr_embeds_per_file': [22, 4, 3, 4, 10, 10, 10]},
 'model_name': 'custom_birdnet',
 'nr_embeds_total': 63,
 'sample_rate (Hz)': 48000,
 'segment_length (samples)': 144000,
 'total_dataset_length (s)': 181.4346875}